# A FAIR² Mental Health Survey Dataset Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the "A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya" using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)

# Access metadata as an object
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Let's list the record sets and their fields (by `@id`) defined in the Croissant schema.

In [ ]:
# List out all record sets and their field ids
record_sets = metadata.record_set
if not record_sets:
    print("No record sets found in metadata. Please check the schema or use mlCroissant's introspection tools.")
else:
    for rs in record_sets:
        print(f"RecordSet: {rs['@id']}")
        print("  Fields:")
        for f in rs['field']:
            print(f"    Field @id: {f['@id']} (name: {f.get('name','')})")
        print()

Now, let's preview the first few records from one record set using its `@id`.

In [ ]:
# Preview the records from the main survey record set
# Replace with the main RecordSet @id from the overview above
main_record_set_id = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2/recordset/mental_health_survey'
records_iterator = dataset.records(record_set=main_record_set_id)

for i, record in enumerate(records_iterator):
    print(record)
    if i > 2:
        break  # Limit output to first 3 records

## 3. Data Extraction
Load data from the survey record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract all data from the main survey record set
record_sets_ids = [main_record_set_id]
dataframes = {}

for record_set_id in record_sets_ids:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)

print(f"Survey columns: {dataframes[main_record_set_id].columns.tolist()}")
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This may include removing outliers, transforming data distributions, or grouping by key demographic attributes.

We'll use the GAD-7 score (generalized anxiety disorder), PHQ-9 score (depression), or PSQ score (perceived stress) for numeric analysis.

In [ ]:
# Assume GAD-7 score column is named by its field @id
# Adjust to actual @id from the schema, e.g., gad7_score_field_id
gad7_score_field_id = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2/field/gad7_score'

survey_df = dataframes[main_record_set_id]

if gad7_score_field_id in survey_df.columns:
    threshold = 10  # Moderate/severe anxiety cut-off
    filtered_df = survey_df[survey_df[gad7_score_field_id] > threshold]
    print(f"Filtered records with GAD-7 score > {threshold}:")
    print(filtered_df.head())

    # Normalize the GAD-7 score
    filtered_df[f"{gad7_score_field_id}_normalized"] = (filtered_df[gad7_score_field_id] - filtered_df[gad7_score_field_id].mean()) / filtered_df[gad7_score_field_id].std()
    print(f"Normalized GAD-7 score for filtered records:")
    print(filtered_df[[gad7_score_field_id, f"{gad7_score_field_id}_normalized"].head())

    # Group by demographic field, such as level_of_education
    group_field_id = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2/field/level_of_education'
    if group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[gad7_score_field_id].mean().reset_index()
        print(f"Mean GAD-7 score grouped by Level of Education:")
        print(grouped_df.head())
else:
    print(f"Field {gad7_score_field_id} not found in DataFrame columns. Available columns: {survey_df.columns.tolist()}")

## 5. Visualization
Visualize the distribution of GAD-7 scores and their relationship with demographic attributes (E.g., Level of Education).

We use Seaborn and Matplotlib for quick plots.

In [ ]:
# Visualization: Distribution of GAD-7 scores
if gad7_score_field_id in survey_df.columns:
    plt.figure(figsize=(8,5))
    sns.histplot(survey_df[gad7_score_field_id].dropna(), bins=10, kde=True)
    plt.title('Distribution of GAD-7 Scores')
    plt.xlabel('GAD-7 Score')
    plt.ylabel('Count')
    plt.show()

    # Boxplot by education level
    if group_field_id in survey_df.columns:
        plt.figure(figsize=(10,6))
        sns.boxplot(x=survey_df[group_field_id], y=survey_df[gad7_score_field_id])
        plt.title('GAD-7 Score by Level of Education')
        plt.xlabel('Level of Education')
        plt.ylabel('GAD-7 Score')
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the data exploration process.

- The dataset contains survey responses on mental health indicators including GAD-7, PHQ-9, and demographic data.
- Record sets and fields are referenced using their unique Croissant `@id` attributes, ensuring reproducible access.
- Data is loaded and processed efficiently using `mlcroissant`, facilitating advanced analyses and visualizations.
- Filtering, normalization, and grouping highlight trends in anxiety scores relative to education levels.
- Further analyses can extend to PHQ-9 and PSQ scores or other subgroup explorations for community health insights.